In [1]:
import numpy as np
import pandas as pd

In [2]:
dtype_map = {
    "Task Id": "string",
    "FPOGV": "Int64", "FPOGID": "Int64",
    "LPMMV": "Int64", "RPMMV": "Int64",
    "BKID": "Int64",  "BKDUR": "float64", 
    "AOI": "string"
}
fixations_labeled = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_labeled_with_blocks.csv", dtype=dtype_map, low_memory=False)
gaze_labeled      = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_labeled_with_blocks.csv", dtype=dtype_map, low_memory=False)

In [3]:
def keep_on_off(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return only rows where df['label'] is 'ON' or 'OFF'.
    No phase-based mapping—just strict label filtering.
    """
    if "label" not in df.columns:
        raise KeyError("DataFrame has no 'label' column.")

    out = df.copy()
    out["label"] = out["label"].astype(str).str.strip().str.upper()
    out = out[out["label"].isin(["ON", "OFF"])].copy()

    # quick sanity check
    print("Kept rows:", len(out))
    print(out["label"].value_counts())

    return out

# Example usage
fixations_onoff = keep_on_off(fixations_labeled)
gaze_onoff      = keep_on_off(gaze_labeled)



fixations_onoff.to_csv("outputs/fixations_ON_OFF_only.csv", index=False)
gaze_onoff.to_csv("outputs/gaze_ON_OFF_only.csv", index=False)

Kept rows: 40862
label
ON     28295
OFF    12567
Name: count, dtype: int64
Kept rows: 1212581
label
ON     808805
OFF    403776
Name: count, dtype: int64


In [4]:
TICK_HZ = 10_000_000  # TIMETICK(f=10000000)

# --- Build ONE shared anchor per (Video, User) across any number of tables ---
def build_shared_anchor(*dfs,
                        tick_col: str = "TIMETICK(f=10000000)",
                        group_cols=("Video Name (folder)", "User ID (file)")) -> pd.DataFrame:
    refs = []
    for d in dfs:
        tmp = d[list(group_cols) + [tick_col]].copy()
        tmp[tick_col] = pd.to_numeric(tmp[tick_col], errors="coerce")
        refs.append(tmp.groupby(list(group_cols), as_index=False)[tick_col].min()
                        .rename(columns={tick_col: "min_tick"}))
    ref = (pd.concat(refs, ignore_index=True)
             .groupby(list(group_cols), as_index=False)["min_tick"].min()
             .rename(columns={"min_tick": "anchor_tick"}))
    return ref

# 1) Window ids by TIMETICK (per Video+User), no task-start anchoring
def add_window_id_by_timetick(
    df: pd.DataFrame,
    *,
    win_sec: float = 3.0,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)"),
    shared_ref: pd.DataFrame | None = None,   # << pass build_shared_anchor(...) here for perfect alignment
    out_col: str = "win_id",
) -> pd.DataFrame:
    out = df.copy()
    out[tick_col] = pd.to_numeric(out[tick_col], errors="coerce")
    win_ticks = int(round(win_sec * TICK_HZ))

    if shared_ref is not None:
        # use the SAME anchor_tick for both streams
        out = out.merge(shared_ref, on=list(group_cols), how="left", validate="many_to_one")
        anchor = out["anchor_tick"]
    else:
        # fallback: per-DF min (works, but can shift if streams start at different ticks)
        anchor = out.groupby(list(group_cols))[tick_col].transform("min")

    out[out_col] = ((out[tick_col] - anchor) // win_ticks).astype("Int64")
    return out

# 2) Fixation features (3): count, mean duration, AOI ratio
def window_features_fixations(fix_df: pd.DataFrame) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]

    def _agg(w: pd.DataFrame) -> pd.Series:
        return pd.Series({
            "fix_count": w["FPOGID"].nunique() if "FPOGID" in w else np.nan,
            "fix_mean_dur": w["FPOGD"].mean()   if "FPOGD" in w else np.nan,
            "fix_aoi_ratio": w["AOI"].notna().mean() if "AOI" in w else np.nan,
        })

    return (fix_df.dropna(subset=["win_id"])
                 .groupby(keys, dropna=False)
                 .apply(_agg)
                 .reset_index())

# 3) Gaze + pupil + saccade + blink + quality features (9)
def window_features_gaze(gaze_df: pd.DataFrame) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]

    def _agg(w: pd.DataFrame) -> pd.Series:
        # Validity masks
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)
        lv = (w["LPMMV"] == 1) if "LPMMV" in w else pd.Series(False, index=w.index)
        rv = (w["RPMMV"] == 1) if "RPMMV" in w else pd.Series(False, index=w.index)

        # --- Gaze spatial (2): std over valid gaze samples
        gx_std = w.loc[gv, "FPOGX"].std() if "FPOGX" in w else np.nan
        gy_std = w.loc[gv, "FPOGY"].std() if "FPOGY" in w else np.nan

        # --- Pupil (2): row-wise average over valid eyes, then window mean/var
        if "LPMM" in w and "RPMM" in w:
            l = w["LPMM"].where(lv)
            r = w["RPMM"].where(rv)
            pupil_row = pd.concat([l, r], axis=1).mean(axis=1, skipna=True)
            pupil_mean = pupil_row.mean()
            pupil_var  = pupil_row.var(ddof=1)
        else:
            pupil_mean, pupil_var = np.nan, np.nan

        # --- Saccade (2)
        sac_mag  = w.loc[gv, "SACCADE_MAG"].mean() if "SACCADE_MAG" in w else np.nan
        sac_dirv = w.loc[gv, "SACCADE_DIR"].var(ddof=1) if "SACCADE_DIR" in w else np.nan

        # --- Blink (2)
        blink_count    = w.loc[gv, "BKID"].nunique() if "BKID"  in w else np.nan
        blink_mean_dur = w.loc[gv, "BKDUR"].mean()   if "BKDUR" in w else np.nan

        # --- Quality (1)
        valid_gaze_ratio = gv.mean() if len(w) else np.nan

        return pd.Series({
            "gaze_x_dispersion": gx_std,
            "gaze_y_dispersion": gy_std,
            "pupil_mean": pupil_mean,
            "pupil_var": 0 if pd.isna(pupil_var) else pupil_var,
            "saccade_mean_mag": sac_mag,
            "saccade_dir_var": 0 if pd.isna(sac_dirv) else sac_dirv,
            "blink_count": blink_count,
            "blink_mean_dur": blink_mean_dur,
            "valid_gaze_ratio": valid_gaze_ratio,
        })

    return (gaze_df.dropna(subset=["win_id"])
                 .groupby(keys, dropna=False)
                 .apply(_agg)
                 .reset_index())

# 4) Merge fixation + gaze features (keep ON/OFF/phase separate)
def combine_fix_gaze_windows(fix_w: pd.DataFrame, gaze_w: pd.DataFrame) -> pd.DataFrame:
    keys = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]
    merged = pd.merge(fix_w, gaze_w, on=keys, how="outer", validate="one_to_one")
    return merged

In [5]:
anchor = build_shared_anchor(fixations_onoff, gaze_onoff)
fixations_win = add_window_id_by_timetick(fixations_onoff, win_sec=3.0, shared_ref=anchor)
gaze_win      = add_window_id_by_timetick(gaze_onoff,      win_sec=3.0, shared_ref=anchor)

In [6]:
fix_w = window_features_fixations(fixations_win)
gaz_w = window_features_gaze(gaze_win)
windows = combine_fix_gaze_windows(fix_w, gaz_w)

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_99424/764406834.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_99424/764406834.py:108: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_agg)


In [ ]:
QC_MIN_VALID = 0.5   # try 0.5–0.7; adjust after seeing counts
keep = (windows["valid_gaze_ratio"] >= QC_MIN_VALID)
windows_qc = windows.loc[keep].copy()
print("QC kept:", len(windows_qc), " / ", len(windows))

In [ ]:
windows_qc.to_csv("outputs/windows_requested_features.csv", index=False)

drop not useful windows

In [ ]:
KEY = ["Video Name (folder)","User ID (file)","Task Id","type","label","phase","block_id","win_id"]
assert windows_qc.duplicated(KEY).sum() == 0
print(windows_qc["label"].value_counts())
print(windows_qc["type"].value_counts())
print(windows_qc["valid_gaze_ratio"].describe())
print(windows_qc[["fix_count","fix_mean_dur","gaze_x_dispersion","pupil_mean"]].describe())